### **Connect Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Import Libraries & Install Dependencies**

In [ ]:
import re
import pandas as pd
import numpy as np
from tqdm import tqdm

### **Import Dataset**

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Datasets/combined_augmented_codemixed_dataset.csv')
print(df.head())

   id                                               text  label language  \
0   1  gue literally belum makan dr tadi, starving bg...      0    mixed   
1   2  they keep acting like they're superior, padaha...      1    mixed   
2   3  ngl gue lowkey baper pas dia tiba2 so sweet gi...      0    mixed   
3   4  semua perempuan tuh emang manipulative, gak ad...      1    mixed   
4   5  deadline besok tp gue blm buka materi, i think...      0    mixed   

   has_slang  has_abbrev     source  
0          0           0  generated  
1          0           0  generated  
2          0           0  generated  
3          0           0  generated  
4          0           0  generated  


In [ ]:
print(df.columns.tolist())

['id', 'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source']


### **Word List — Bahasa Indonesia**

In [ ]:
SLANG_WORDS_ID = {
    # -- A1. Kata ganti (pronoun) informal
    "gue", "gw", "gua",
    "lo", "lu", "elo", "elu", "loe", "lw",
    "doi", "die", "dye",
    "kite",                     # kita (Betawi)
    "ane", "ente",              # saya / kamu (prokem Arab-Betawi)
    "akika",                    # aku (bahasa banci/gay slang)
    "aq",                       # aku (alay)
    "qt",                       # kita (alay)
    "dy",                       # dia (alay)

    # -- A2. Tawa & ekspresi imitatif
    "wkwk", "wkwkwk", "wkwkwkwk", "wkwkwkwkwk",
    "wkakak", "kwkwk",
    "xixi", "xixixi",
    "hehe", "hihi", "huhu", "hoho",
    "hahahah",
    "hiks", "hikz",
    "zzz", "uwu", "owo",

    # -- A3. Sapaan & panggilan
    "cuy", "coy", "cok",
    "gaes", "lur", "sob",
    "beb", "beby", "say",
    "bestie", "bff",
    "anjay", "anjir", "njir", "njay",
    "astaga", "astg",
    "bokap", "nyokap", "bonyok",
    "cewek", "cewe", "cowok", "cowo",
    "bang", "abang",
    "kak", "kaka", "ka", "kk",
    "brur", "ngab",

    # -- A4. Fresh & Creative (kata gaul kreatif)
    "baper",     # bawa perasaan
    "mager",     # malas gerak
    "gabut",     # ga ada buat
    "bucin",     # budak cinta
    "julid",     # nyinyir/iri
    "kepo",      # penasaran
    "santuy",    # santai (walikan)
    "kuy",       # yuk (walikan)
    "sabi",      # bisa (walikan)
    "kece",      # keren (gaul)
    "gokil",     # gila/keren
    "alay",      # norak/lebay
    "lebay",     # berlebihan
    "bete",      # kesal
    "galau",     # bimbang/sedih
    "mantul",    # mantap betul
    "mantap",    # bagus sekali (gaul)
    "asoy",      # asyik (Sunda gaul)
    "slebew",    # ekspresi kagum Gen-Z 2023+
    "gaskeun",   # ayo lakukan
    "gaspol",
    "gemoy",     # menggemaskan (2023-2024)
    "gemes",
    "cemungud",  # semangat (alay)
    "sultan",    # orang kaya (gaul)
    "cuan",      # uang/keuntungan
    "gabisa",    # tidak bisa
    "gamau",     # tidak mau
    "gausah",    # tidak usah
    "yaman",     # lumayan (walikan)
    "jamet",     # jawa metal -> uncool
    "jomblo", "jomlo",
    "gebetan",   # orang yang ditaksir
    "naksir",    # suka seseorang
    "modus", "healing",
    "bobok",     # tidur (gaul)
    "caper",     # cari perhatian
    "minder", "pamer",
    "emosi",     # marah (gaul)
    "oce",       # oke
    "receh",     # remeh
    "gercep",    # gerak cepat
    "pansos",    # panjat sosial
    "ngaret",    # terlambat/molor

    # -- A5. Flippant (santai/meremehkan)
    "garing", "jayus", "cuek", "cupu", "culun", "dodol",
    "telmi",     # telat mikir
    "kuper",     # kurang pergaulan
    "narsis", "centil",
    "jijay", "jijik",
    "bawel", "ngoceh", "nyinyir", "nyebelin", "ngeselin",

    # -- Sosmed & platform
    "netizen", "warganet",
    "ngehype", "ngetrend", "ngeviral",

    # -- Emosi & mental state (sosmed/informal)
    "toxic", "vibes", "overthinking", "triggered",
    "insecure", "burnout",
    "gaslight", "gaslighting",
    "manipulative", "narcissist", "redflag",
    "overwhelmed", "drained", "numb", "spiraling",
    "doomscrolling", "hyperfixation", "ick",

    # -- Penilaian & sikap (Gen-Z/sosmed)
    "cringe", "cringey", "valid", "slay", "iconic",
    "aesthetic", "wholesome", "petty", "shady", "salty",
    "extra", "basic", "lowkey", "highkey", "mid", "based",
    "ratio", "ratioed", "sus", "savage", "badass",
    "loser", "wannabe", "fake",
    "overrated", "underrated", "overhyped", "washed",
    "delusional", "delulu", "unhinged", "chaotic",
    "obsessed", "clingy", "possessive",
    "cooked", "flopped", "snatched",
    "npc", "clowned", "goated", "sigma", "copium", "mogged",

    # -- Hubungan & sosial
    "ghosting", "ghosted", "crush", "squad",
    "fomo", "yolo", "hangout", "dating", "flirting",
    "friendzone", "friendzoned", "situationship",
    "rizz", "rizzing", "rizzy", "rizzler",
    "simp", "simping", "pickme", "bodycount",
    "girlboss", "boysober", "benching",
    "breadcrumbing", "lovebombing", "glowup",

    # -- Aktivitas digital
    "hype", "catfish", "catfished",
    "troll", "trolling", "doxing", "doxxing",
    "spam", "spamming", "bot", "astroturfing",
    "shadowbanned", "clout", "flexing", "flex",
    "cancel", "cancelled", "canceling", "canceled",
    "exposed", "exposing",

    # -- Gen-Z internet culture
    "nocap", "cap", "capping", "periodt",
    "tea", "spill", "shade", "pookie", "era",
    "bet", "bussin", "sheesh",
    "gatekeep", "gatekeeping",
    "brat", "demure", "girlypop",
}


In [ ]:
SLANG_PHRASES_ID = {
    # -- Frasa gaul Indonesia asli
    "ga ada otak", "gak ada otak",
    "dasar lo",
    "emang lo siapa",
    "mau apa lo",
    "gabut banget",
    "gak jelas",
    "ga jelas",
    "pdkt sama",

    # -- Frasa code-mixed (Inggris dalam konteks Indonesia)
    "no cap",
    "red flag", "green flag", "beige flag",
    "main character", "main character energy",
    "understood the assignment",
    "touch grass",
    "skill issue",
    "rent free", "living rent free",
    "girl math", "girl dinner",
    "chronically online",
    "pick me", "pick me girl",
    "friend zone",
    "love bombing",
    "talking stage",
    "soft launch", "hard launch",
    "say less",
    "on god",
    "very demure", "very mindful",
    "roman era",
    "brat summer",
    "caught in 4k",
    "ate and left no crumbs", "left no crumbs",
    "its giving", "it's giving",
    "plot twist",
    "character development",
    "girl boss",
    "vibe check",
    "gassed up",
    "body count",
    "glow up",
    "not me",
    "for real for real",
    "hits different",
    "gaslight girlboss gatekeep",
    "this ain't it",
    "and i oop",
}


In [ ]:
ABBREV_WORDS_ID = {
    # -- Kata ganti & kata tanya
    "yg", "sy", "km", "mrk", "kpn",
    "dmn", "dmana",
    "gmn", "knp", "spy", "sp",
    "skrg", "skrang",
    "stlh", "sblm",
    "sbnrnya", "sbnarnya", "sebenrnya", "sbenrnya",
    "mksdnya", "kyknya", "pknnya", "sbg",
    "smua", "stiap", "mgkin", "kmn",
    "drpd", "ttpi", "tnpa", "smpe",
    "pdhl", "wlpn", "mnrt", "bgmn", "bgitu", "sklpn",

    # -- Negasi
    "gak", "ga", "g", "gk",
    "ngga", "nggak", "ndak", "kagak", "enggak",
    "gapapa", "gpp",
    "gamau", "gamu",
    "gabisa", "gabsa",
    "gausah",
    "gabtau", "gatau", "gtw",
    "gasuka", "gasih",
    "gaada", "ganiat",
    "bkn", "blm", "jgn",
    "krg", "lbh", "bnyk", "sdkt",

    # -- Kata kerja & status
    "udh", "udah", "sdh", "dah",
    "msh", "mo", "bs", "hrs", "brg",
    "pg", "dtg", "plng", "tdr", "mkn", "mnm",
    "blj", "krj", "ntn", "klr", "msk",
    "jwb", "tny", "tlg", "bwt",
    "pke", "pk", "pny",
    "byar", "byr", "kluar", "dpt",
    "brkt", "ktmu",
    "nyoba", "ngerjain", "balikin", "diceritain", "nemenin",

    # -- Partikel & konjungsi disingkat
    "jg",
    "aj", "aja",
    "sm", "tp", "tpi",
    "krn", "karna", "krna",
    "klo", "klu",
    "sprt", "spt",
    "ato", "kyk", "trs", "trus",
    "jd", "jdi", "pd",
    "dr", "dri",
    "dgn", "dg", "dngn",
    "dlm", "utk", "ttg", "thd",
    "ttap", "brrti", "bkl",

    # -- Preposisi singkat
    "d",                            # di
    "k",                            # ke

    # -- Intensifier disingkat
    "bgt", "bngt", "bngd",          # banget
    "sgt", "plg",
    "bnr", "bner",

    # -- Waktu & tempat
    "lg", "lgi", "td", "dlu",
    "dpn", "blkg", "bwh", "ats",
    "sna", "sni", "sblah", "mlm",
    "kmren", "kmarin",
    "bsok", "bsk",
    "mgu", "mgg", "bln", "thn", "jm", "malem",

    # -- Kata benda umum
    "org",
    "tmn", "tmen",
    "klg", "hp", "nmr",
    "jln", "jl",
    "tggl", "tgl",
    "kmps", "mhsw", "dsn", "rmh",
    "kmpung", "mbl", "mtr", "pkt", "ktr",

    # -- Kota (singkatan informal)
    "jkt", "sby", "bdg", "mdn",
    "yk", "jogja",
    "bks", "dpk", "tgr", "smr", "mksr", "plbg",

    # -- Platform & aplikasi
    "wa",
    "ig", "insta",
    "tt", "yt", "ytb", "fb",
    "tele", "tg", "twt", "sptfy",
    "dm", "pm", "dms",

    # -- Singkatan formal yang dipakai informal
    "dll", "dkk", "dst", "dsb", "tsb",

    # -- Respons cepat (Indonesia)
    "ok", "oke", "oce",
    "mksh", "mks",
    "mf", "maap",
    "sori", "sry",
    "pls", "plz",
    "hbs", "jls", "smngt", "otw", "pdkt",

    # -- Singkatan Inggris umum dalam code-mixed Indonesia
    "thx", "tks", "tq", "ty", "tnx",
    "lmk", "omw", "wdym", "wbu", "hbu",
    "jk", "jkjk", "nvm", "yw", "hmu",
    "ttyl", "gtg", "g2g", "bbl", "brb",
    "omg", "wtf", "wth", "rofl",
    "lol", "lmao", "lmfao", "omfg",
    "lmaooo", "lolll",
    "asap", "pov", "tw", "cw",
    "iykyk", "tbt", "ootd", "grwm",
    "fr", "frfr", "ong", "istg",
    "idc", "idk", "smh", "ngl",
    "np", "btw", "fyi", "imo", "imho", "ikr", "tbh",
    "ily", "ilysm", "tysm",
    "stfu", "kys", "kms",          # toxic abbreviations
    "idgaf", "wyd", "wya", "sup",
    "icymi", "js", "gg", "afk", "irl",
    "nsfw", "sfw",

    # -- Reduplikasi alay (juga dideteksi via regex)
    "jalan2", "makan2", "temen2", "main2",
    "santai2", "jln2", "malem2", "pagi2",
    "diem2", "lama2",
}


### **Word List — Bahasa Inggris**

In [ ]:
SLANG_WORDS_EN = {
    # -- Gen-Z core slang
    "slay", "slays", "slaying",
    "rizz", "rizzing", "rizzy", "rizzler",
    "delulu", "bussin", "sheesh", "periodt",
    "nocap", "cap", "capping",
    "lowkey", "highkey", "mid",
    "sus",
    "vibe", "vibes", "vibing",
    "based", "ratio", "ratioed",
    "snatched", "goated", "npc", "slaps",
    "brat", "demure", "girlypop",
    "pookie", "era", "yeet", "bet",
    "cringe", "cringy", "cringey",
    "naur", "sksksk", "oomf", "moots", "bffr",
    "boomer", "karen", "chad", "incel",
    "gaslit", "glowup", "copium", "sigma", "mogged",
    "gatekeep", "gatekeeping",
    "fax", "hella", "deadass",

    # -- Digital relationship & social slang
    "ghosting", "ghosted",
    "situationship",
    "simp", "simping",
    "pickme",
    "friendzone", "friendzoned",
    "girlboss", "girlbossing",
    "boysober", "benching",
    "breadcrumbing", "lovebombing",
    "redflag",
}


In [ ]:
SLANG_PHRASES_EN = {
    "no cap",
    "it's giving", "its giving",
    "hits different",
    "main character", "main character energy",
    "touch grass",
    "skill issue",
    "on god",
    "for real for real",
    "say less",
    "understood the assignment",
    "left no crumbs",
    "rent free", "living rent free",
    "the audacity",
    "sending me", "this is sending me",
    "and i oop",
    "plot twist",
    "slay queen",
    "girl math", "girl dinner",
    "character development",
    "gaslight girlboss gatekeep",
    "not me",
    "bestie behavior",
    "red flag", "green flag", "beige flag",
    "soft launch", "hard launch",
    "talking stage",
    "body count",
    "love bombing",
    "dm slide",
    "soft life",
    "girl boss",
    "pick me",
    "friend zone",
    "gassed up",
    "vibe check",
    "roman era", "very demure",
    "chronically online",
    "attention seeker",
    "brat summer",
    "caught in 4k",
    "this ain't it",
    "ate and left no crumbs",
    "i'm crying",
}


In [ ]:
ABBREV_WORDS_EN = {
    # -- Letter substitution (texting-era)
    "u", "r", "ur",
    "gr8", "l8", "l8r", "b4",
    "h8", "m8", "sk8",
    "abt", "bc", "rn",
    "imo", "ngl", "tbh", "imho",
    "afaik", "afk", "irl", "tldr",

    # -- Texting abbreviations
    "omw", "otw", "smh",
    "istg", "idk", "idc", "idgaf",
    "wdym", "wdyt", "wbu", "hbu",
    "fyi", "btw",
    "jk", "jkjk", "nvm",
    "np", "yw", "tysm",
    "ily", "ilysm", "ikr",
    "lmk", "hmu",
    "ttyl", "ttys",
    "gtg", "g2g",
    "bbl", "brb", "brt",
    "omg", "wtf", "wth",
    "rofl", "lol", "lmao", "lmfao",
    "asap", "tbt", "ootd", "grwm",
    "iykyk", "fomo", "yolo", "goat", "pov",
    "tw", "cw", "icymi", "fwiw",
    "mfw", "tfw", "nbd", "atm",
    "til", "eli5", "ftfy",
    "glhf", "gratz", "iirc", "op",
    "nsfw", "sfw", "gg", "tbf", "nts",

    # -- Platform abbreviations
    "ig", "tt", "yt", "fb", "twt",
    "dm", "pm", "dms",

    # -- Quick response
    "thx", "tks", "tq", "ty", "tnx",
    "pls", "plz", "wya", "wyd",
}


### **Combined Word List (ID + EN)**

In [ ]:
SLANG_WORDS_COMBINED   = SLANG_WORDS_ID   | SLANG_WORDS_EN
SLANG_PHRASES_COMBINED = SLANG_PHRASES_ID | SLANG_PHRASES_EN
ABBREV_WORDS_COMBINED  = ABBREV_WORDS_ID  | ABBREV_WORDS_EN

# Cek overlap antar komponen
overlap_sl_ab = SLANG_WORDS_COMBINED & ABBREV_WORDS_COMBINED
if overlap_sl_ab:
    print(f"[INFO] Overlap SLANG_WORDS o ABBREV_WORDS ({len(overlap_sl_ab)} kata): {sorted(overlap_sl_ab)}")
    print("       Overlap disengaja: toxic markers / kata yang berfungsi ganda.")

print(f"[INFO] SLANG_WORDS_ID      : {len(SLANG_WORDS_ID):>4} kata")
print(f"[INFO] SLANG_WORDS_EN      : {len(SLANG_WORDS_EN):>4} kata")
print(f"[INFO] SLANG_WORDS_COMBINED: {len(SLANG_WORDS_COMBINED):>4} kata (union)")
print()
print(f"[INFO] SLANG_PHRASES_ID      : {len(SLANG_PHRASES_ID):>4} frasa")
print(f"[INFO] SLANG_PHRASES_EN      : {len(SLANG_PHRASES_EN):>4} frasa")
print(f"[INFO] SLANG_PHRASES_COMBINED: {len(SLANG_PHRASES_COMBINED):>4} frasa (union)")
print()
print(f"[INFO] ABBREV_WORDS_ID      : {len(ABBREV_WORDS_ID):>4} kata")
print(f"[INFO] ABBREV_WORDS_EN      : {len(ABBREV_WORDS_EN):>4} kata")
print(f"[INFO] ABBREV_WORDS_COMBINED: {len(ABBREV_WORDS_COMBINED):>4} kata (union)")


[INFO] Overlap SLANG_WORDS o ABBREV_WORDS (6 kata): ['fomo', 'gabisa', 'gamau', 'gausah', 'oce', 'yolo']
       Overlap disengaja: toxic markers / kata yang berfungsi ganda.
[INFO] SLANG_WORDS_ID      :  264 kata
[INFO] SLANG_WORDS_EN      :   72 kata
[INFO] SLANG_WORDS_COMBINED:  285 kata (union)

[INFO] SLANG_PHRASES_ID      :   54 frasa
[INFO] SLANG_PHRASES_EN      :   51 frasa
[INFO] SLANG_PHRASES_COMBINED:   63 frasa (union)

[INFO] ABBREV_WORDS_ID      :  314 kata
[INFO] ABBREV_WORDS_EN      :  105 kata
[INFO] ABBREV_WORDS_COMBINED:  349 kata (union)


### **Minimal Clean**

In [ ]:
def minimal_clean(text):
    if pd.isna(text):
        return ""
    t = str(text).strip()
    # Curly quotes & common Unicode artifacts
    t = (
        t.replace("\u2018", "'")
         .replace("\u2019", "'")
         .replace("\u201c", '"')
         .replace("\u201d", '"')
    )
    t = t.replace("&lt;",   " ") \
     .replace("&gt;",   " ") \
     .replace("&nbsp;", " ") \
     .replace("&apos;", "'") \
     .replace("&quot;", '"')
    # HTML entities
    t = re.sub(r"&#\d+;", " ", t)
    # Latin-1 / Windows-1252 artifacts
    t = re.sub(r"[\xc3\xc2\xe2][^\s]*", " ", t)
    t = re.sub(r"\\x[0-9a-fA-F]{2}", "", t)
    # Whitespace normalization
    t = re.sub(r"\s+", " ", t).strip()
    return t


### **Pengecekan & Koreksi Fitur (has\_slang & has\_abbrev)**

In [ ]:
def _check_slang(text: str) -> int:
    text_lower = str(text).lower()

    # (a) Single-token whole-word matching
    tokens = set(re.findall(r'\b\w+\b', text_lower))
    if tokens & SLANG_WORDS_COMBINED:
        return 1

    # (b) Multi-word phrase substring matching
    for phrase in SLANG_PHRASES_COMBINED:
        if phrase in text_lower:
            return 1

    # (c) Indonesian nge- verbal prefix (distinctly informal)
    #     Pola: nge + kata kerja >= 3 karakter
    if re.search(r'\bnge[a-z]{3,}\b', text_lower):
        return 1

    return 0


def _check_abbrev(text: str) -> int:
    text_lower = str(text).lower()
    tokens = set(re.findall(r'\b\w+\b', text_lower))

    # (a) Kamus abbrev (whole-word match)
    if tokens & ABBREV_WORDS_COMBINED:
        return 1

    # (b) Leet-speak: huruf + digit + huruf (gr8, l8r, b4, 4nj1ng, g2g)
    if re.search(r'\b(?:[a-z]+[0-9][a-z0-9]*|[0-9]+[a-z][a-z0-9]*)\b', text_lower):
        return 1

    # (c) Reduplikasi alay Indonesia: kata + angka 2 (jalan2, temen2)
    if re.search(r'[a-z]{2,}2(?:\b|$)', text_lower):
        return 1

    return 0

### **Full Cleaning**

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    t = str(text).strip().lower()

    # Encoding fix
    t = re.sub(r"\\[ntr]", " ", t)
    t = re.sub(r"(\\\s*)+", " ", t)
    t = t.replace("&amp;", " and ")
    # Elongation: maks 2 karakter berulang (anjiiir => anjir)
    t = re.sub(r"([^.!?])\1{2,}", r"\1\1", t)

    # Sisa encoding noise
    t = re.sub(r"\bx[0-9]{2,3}\b", " ", t)
    t = re.sub(r"\b\w*[\xc3\xc2\xe2][\w\xc3\xc2\xe2]*\b", " ", t)
    t = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", t)

    # Twitter/X artifacts: RT
    t = re.sub(r"(?i)^rt\s+", "", t)
    t = re.sub(r"(?i)\brt\b", " ", t)
    t = re.sub(r"(?i)\bretweeted\b.*?\buser\b\)?\s*[:;]*", " ", t)

    t = re.sub(r"@\w+", " user ", t)

    # Hapus URL
    t = re.sub(r"http\S+|www\.\S+|t\.co/\S+", " ", t)

    # Hashtag: hapus #, pertahankan teks
    t = re.sub(r"#(\w+)", r"\1", t)

    # Hapus emoji & karakter di luar BMP
    t = re.sub(r"[\U00010000-\U0010ffff]", " ", t)

    # Hapus angka berdiri sendiri (bukan bagian leet-speak)
    # '4nj1ng': 4 diikuti huruf -> TIDAK dihapus
    # 'meet at 4': 4 diapit spasi -> DIHAPUS
    t = re.sub(r"(?<![a-z])\d+(?![a-z])", " ", t)

    # Tanda petik berlebih
    t = t.replace('"', " ")
    t = re.sub(r"(?<![a-zA-Z])'", " ", t)
    t = re.sub(r"'(?![a-zA-Z])", " ", t)

    # Tanda baca berulang
    t = re.sub(r"\?{2,}", " ?", t)
    t = re.sub(r"!{2,}", " !", t)
    t = re.sub(r"\.{4,}", "...", t)
    t = re.sub(r"[:;]{2,}", " ", t)

    # Deduplikasi 'user user' => 'user'
    t = re.sub(r"(?:\buser\b\s*){2,}", "user ", t)

    # Hapus karakter non-alfanumerik kecuali yang relevan
    # Angka dipertahankan agar leet-speak tidak rusak
    t = re.sub(r"[^a-zA-Z0-9\s',.?!]", " ", t)

    # Sisa apostrof gantung
    t = re.sub(r"(?<![a-zA-Z])'", " ", t)
    t = re.sub(r"'(?![a-zA-Z])", " ", t)

    # Normalize whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t


### **Pipeline**

In [ ]:
def run_pipeline_mixed(
    input_path:  str,
    output_path: str,
    text_col:    str  = "text",
    label_col:   str  = "label",
    slang_col:   str  = "has_slang",
    abbrev_col:  str  = "has_abbrev",
    source:      str  = "generated",
    seed:        int  = 42
):
    LANGUAGE     = "mixed"
    DRIFT_THRESH = 0.05      
    SEP_MAIN = "=" * 62
    SEP_SUB  = "-" * 55

    print(f"\n{SEP_MAIN}")
    print(f"  PIPELINE: CODE-MIXED")
    print(f"{SEP_MAIN}")
    print(f"\n[1] LOAD RAW DATA")
    df = pd.read_csv(input_path)
    before_na = len(df)
    print(f"    File            : {input_path}")
    print(f"    Baris dimuat    : {before_na:>7,}")
    print(f"    Kolom           : {list(df.columns)}")

    # Pastikan kolom yang dibutuhkan tersedia
    for col in [text_col, label_col, slang_col, abbrev_col]:
        if col not in df.columns:
            raise ValueError(
                f"[ERROR] Kolom '{col}' tidak ditemukan. "
                f"Kolom tersedia: {list(df.columns)}"
            )

    df = df[[text_col, label_col, slang_col, abbrev_col]].copy()
    df = df.dropna(subset=[text_col, label_col])
    print(f"    Hapus missing (text/label): {before_na - len(df)}")

    df = df.rename(columns={
        text_col:   "text",
        label_col:  "label",
        slang_col:  "has_slang",
        abbrev_col: "has_abbrev",
    })

    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    before_nb = len(df)
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)
    df = df[df["label"].isin([0, 1])].reset_index(drop=True)
    print(f"    Hapus non-binary: {before_nb - len(df)}")

    for feat_col in ["has_slang", "has_abbrev"]:
        df[feat_col] = pd.to_numeric(df[feat_col], errors="coerce").fillna(0).astype(int)
        df[feat_col] = df[feat_col].clip(0, 1)

    df["text"] = df["text"].astype(str).str.strip()
    before_empty = len(df)
    df = df[df["text"] != ""].reset_index(drop=True)
    print(f"    Hapus kosong    : {before_empty - len(df)}")
    print(f"    Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")
    print(f"    has_slang (LLM) : {df['has_slang'].sum():,} ({df['has_slang'].mean()*100:.1f}%)")
    print(f"    has_abbrev (LLM): {df['has_abbrev'].sum():,} ({df['has_abbrev'].mean()*100:.1f}%)")
    print(f"    Total valid     : {len(df):,} baris")
    n_after_load = len(df)

    print(f"\n[2] MINIMAL CLEANING (encoding fix + normalisasi whitespace)")
    tqdm.pandas(desc="    minimal_clean  ")
    df["_text_mc"] = df["text"].progress_apply(minimal_clean)
    print(f"    Dilakukan pada  : {len(df):,} baris")
    print(f"    Contoh output   : {repr(df['_text_mc'].iloc[0][:80])}")
    n_after_minimal = len(df)

    print(f"\n[3] PENGECEKAN & KOREKSI FITUR (has_slang | has_abbrev)")
    print(f"    Koreksi: LLM=0 -> 1 jika word list mendeteksi (false negative).")
    print(f"    LLM=1 tetap 1 walaupun word list tidak menemukan (trust LLM).")

    tqdm.pandas(desc="    check_slang    ")
    df["_wl_slang"]  = df["_text_mc"].progress_apply(_check_slang)
    tqdm.pandas(desc="    check_abbrev   ")
    df["_wl_abbrev"] = df["_text_mc"].progress_apply(_check_abbrev)

    for feat, wl_col in [("has_slang", "_wl_slang"), ("has_abbrev", "_wl_abbrev")]:
        match    = ((df[feat]==1) & (df[wl_col]==1)).sum()
        llm_only = ((df[feat]==1) & (df[wl_col]==0)).sum()
        wl_only  = ((df[feat]==0) & (df[wl_col]==1)).sum()
        neither  = ((df[feat]==0) & (df[wl_col]==0)).sum()
        print(f"\n    [{feat}] — sebelum koreksi")
        print(f"      LLM=1, WL=1 (confirmed)    : {match:,}")
        print(f"      LLM=1, WL=0 (LLM-only)     : {llm_only:,}  <- di luar word list, tetap 1")
        print(f"      LLM=0, WL=1 (false neg LLM): {wl_only:,}  <- akan dikoreksi ke 1")
        print(f"      LLM=0, WL=0 (neither)       : {neither:,}")

    n_corrected_slang  = ((df["has_slang"]==0) & (df["_wl_slang"]==1)).sum()
    n_corrected_abbrev = ((df["has_abbrev"]==0) & (df["_wl_abbrev"]==1)).sum()
    df["has_slang"]  = (df["has_slang"]  | df["_wl_slang"] ).astype(int)
    df["has_abbrev"] = (df["has_abbrev"] | df["_wl_abbrev"]).astype(int)
    print(f"\n    Koreksi has_slang (0->1)  : {n_corrected_slang:,} baris")
    print(f"    Koreksi has_abbrev (0->1) : {n_corrected_abbrev:,} baris")

    df = df.drop(columns=["_wl_slang", "_wl_abbrev"])

    n_sl = df["has_slang"].sum()
    n_ab = df["has_abbrev"].sum()
    n11  = ((df["has_slang"]==1) & (df["has_abbrev"]==1)).sum()
    n10  = ((df["has_slang"]==1) & (df["has_abbrev"]==0)).sum()
    n01  = ((df["has_slang"]==0) & (df["has_abbrev"]==1)).sum()
    n00  = ((df["has_slang"]==0) & (df["has_abbrev"]==0)).sum()
    print(f"\n    Distribusi setelah koreksi:")
    print(f"    has_slang        : {n_sl:,} ({df['has_slang'].mean()*100:.1f}%)")
    print(f"    has_abbrev       : {n_ab:,} ({df['has_abbrev'].mean()*100:.1f}%)")
    print(f"    (slang, abbrev)  : (1,1)={n11}  (1,0)={n10}  (0,1)={n01}  (0,0)={n00}")
    if df["has_slang"].mean() > 0.95:
        print("    [!] PERINGATAN: has_slang rate > 95% — review SLANG_WORDS (kemungkinan false positive).")
    n_after_feature = len(df)

    print(f"\n[4] DISTRIBUTION ANALYSIS (dicatat untuk Bab 3)")
    print(f"    {SEP_SUB}")
    for lbl_val, lbl_name in [(0, "NON-HATE"), (1, "HATE   ")]:
        sub  = df[df["label"] == lbl_val]
        n_s  = sub["has_slang"].sum()
        n_a  = sub["has_abbrev"].sum()
        n_11 = ((sub["has_slang"]==1) & (sub["has_abbrev"]==1)).sum()
        n_00 = ((sub["has_slang"]==0) & (sub["has_abbrev"]==0)).sum()
        pct  = max(len(sub), 1)
        print(f"    {lbl_name} (n={len(sub):6,}): "
              f"slang={n_s}({n_s/pct*100:.1f}%)  "
              f"abbrev={n_a}({n_a/pct*100:.1f}%)  "
              f"(1,1)={n_11}  (0,0)={n_00}")
    print(f"    {SEP_SUB}")
    label_dist  = df["label"].value_counts(normalize=True).sort_index().to_dict()
    slang_rate  = df["has_slang"].mean()
    abbrev_rate = df["has_abbrev"].mean()
    print(f"    Proporsi label  : {label_dist}")
    print(f"    Rate has_slang  : {slang_rate:.4f}")
    print(f"    Rate has_abbrev : {abbrev_rate:.4f}")

    print(f"\n[5] FULL DATA CLEANING")

    before_dup = len(df)
    df = df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
    n_after_dedup_raw = len(df)
    print(f"    [5a] Dedup teks mentah        : {before_dup:,} => {n_after_dedup_raw:,} "
          f"(hapus {before_dup - n_after_dedup_raw})")

    print("    [5b] Cleaning teks...")
    tqdm.pandas(desc="         clean_text     ")
    df["_text_clean"] = df["_text_mc"].progress_apply(clean_text)
    n_after_clean = len(df)
    print(f"         Selesai               : {n_after_clean:,} baris")

    media_only_pat = r"(?i)\bmedia\s*only\b.*\bno\s*text\b"
    before_postcl = len(df)
    df = df[df["_text_clean"].str.strip().ne("")]
    df = df[~df["_text_clean"].str.contains(media_only_pat, regex=True, na=False)]
    df = df[df["_text_clean"].str.contains(r"[a-zA-Z0-9]", regex=True, na=False)]   
    df = df.drop_duplicates(subset=["_text_clean"], keep="first").reset_index(drop=True)
    n_after_postcl = len(df)
    print(f"    [5c] Filter kosong + dedup   : {before_postcl:,} => {n_after_postcl:,} "
          f"(hapus {before_postcl - n_after_postcl})")

    df["_token_len"] = df["_text_clean"].apply(lambda t: len(t.split()))
    before_len = len(df)
    df = df[(df["_token_len"] >= 3) & (df["_token_len"] <= 512)]
    df = df.drop(columns=["_token_len"]).reset_index(drop=True)
    n_after_lenfilter = len(df)
    print(f"    [5d] Filter panjang (3-512)  : {before_len:,} => {n_after_lenfilter:,} "
          f"(hapus {before_len - n_after_lenfilter})")

    print(f"\n[6] POST-CLEANING FILTER + VERIFIKASI DISTRIBUSI")

    n_final = len(df)
    print(f"    Total baris final: {n_final:,}")
    print(f"    Distribusi label : {df['label'].value_counts().sort_index().to_dict()}")

    post_label_dist  = df["label"].value_counts(normalize=True).sort_index().to_dict()
    post_slang_rate  = df["has_slang"].mean()
    post_abbrev_rate = df["has_abbrev"].mean()
    print(f"\n    Verifikasi drift distribusi (threshold delta > {DRIFT_THRESH:.0%})")
    print(f"    {SEP_SUB}")
    drifted = False
    for lbl_val in [0, 1]:
        pre_v  = label_dist.get(lbl_val, 0)
        post_v = post_label_dist.get(lbl_val, 0)
        delta  = abs(post_v - pre_v)
        flag   = " [!] DRIFT" if delta > DRIFT_THRESH else " ok"
        print(f"    Label={lbl_val}: pre={pre_v:.4f} => post={post_v:.4f}  delta={delta:.4f}{flag}")
        if delta > DRIFT_THRESH:
            drifted = True
    s_delta = abs(post_slang_rate  - slang_rate)
    a_delta = abs(post_abbrev_rate - abbrev_rate)
    flag_s  = " [!] DRIFT" if s_delta > DRIFT_THRESH else " ok"
    flag_a  = " [!] DRIFT" if a_delta > DRIFT_THRESH else " ok"
    print(f"    has_slang : pre={slang_rate:.4f} => post={post_slang_rate:.4f}  delta={s_delta:.4f}{flag_s}")
    print(f"    has_abbrev: pre={abbrev_rate:.4f} => post={post_abbrev_rate:.4f}  delta={a_delta:.4f}{flag_a}")
    if s_delta > DRIFT_THRESH: drifted = True
    if a_delta > DRIFT_THRESH: drifted = True
    print(f"    {SEP_SUB}")
    if drifted:
        print("    [!] Distribusi bergeser signifikan. Pertimbangkan re-generate atau")
        print("        dokumentasikan sebagai limitasi di Bab 3.")
    else:
        print("    ok  Tidak ada drift signifikan. Distribusi terjaga dengan baik.")

    print(f"\n    TABEL TRANSISI DATA (Bab 3 -- Tabel transisi_cleaning)")
    print(f"    {SEP_SUB}")
    print(f"    [1] Data mentah (raw CSV)                         : {before_na:>7,}")
    print(f"    [2] Setelah validasi (NaN / kosong / non-binary)  : {n_after_load:>7,}")
    print(f"    [3] Setelah minimal cleaning                      : {n_after_minimal:>7,}")
    print(f"    [4] Setelah pengecekan & koreksi (has_slang/abbrev): {n_after_feature:>7,}")
    print(f"    [5a] Setelah dedup teks mentah                    : {n_after_dedup_raw:>7,}")
    print(f"    [5b] Setelah full cleaning                         : {n_after_clean:>7,}")
    print(f"    [5c] Setelah post-cleaning filter                 : {n_after_postcl:>7,}")
    print(f"    [5d] Setelah filter panjang token (3-512)         : {n_after_lenfilter:>7,}")
    print(f"    {SEP_SUB}")
    print(f"    FINAL DATASET                                     : {n_final:>7,}")
    print(f"\n[7] SAVE")

    df["id"]       = range(1, len(df) + 1)
    df["language"] = LANGUAGE
    df["source"]   = source

    df_final = pd.DataFrame({
        "id"        : df["id"],
        "text"      : df["_text_clean"], 
        "label"     : df["label"],
        "language"  : df["language"],
        "has_slang" : df["has_slang"],
        "has_abbrev": df["has_abbrev"],
        "source"    : df["source"],
    })
    assert list(df_final.columns) == [
        "id", "text", "label", "language", "has_slang", "has_abbrev", "source"
    ], f"[ERROR] Kolom tidak sesuai: {list(df_final.columns)}"

    df_final.to_csv(output_path, index=False)
    print(f"    [7b] Final CSV   : {output_path}")
    print(f"         Kolom       : {list(df_final.columns)}")

    # Ringkasan akhir
    n11 = ((df_final["has_slang"]==1) & (df_final["has_abbrev"]==1)).sum()
    n10 = ((df_final["has_slang"]==1) & (df_final["has_abbrev"]==0)).sum()
    n01 = ((df_final["has_slang"]==0) & (df_final["has_abbrev"]==1)).sum()
    n00 = ((df_final["has_slang"]==0) & (df_final["has_abbrev"]==0)).sum()
    print(f"\n{SEP_MAIN}")
    print(f"  RINGKASAN AKHIR -- CODE-MIXED")
    print(f"{SEP_MAIN}")
    print(f"  Total baris      : {len(df_final):,}")
    print(f"  Distribusi label : {df_final['label'].value_counts().sort_index().to_dict()}")
    print(f"  has_slang        : {df_final['has_slang'].sum():,} ({df_final['has_slang'].mean()*100:.1f}%)")
    print(f"  has_abbrev       : {df_final['has_abbrev'].sum():,} ({df_final['has_abbrev'].mean()*100:.1f}%)")
    print(f"  (slang, abbrev)  : (1,1)={n11}  (1,0)={n10}  (0,1)={n01}  (0,0)={n00}")
    print(f"  ID range         : {df_final['id'].iloc[0]} - {df_final['id'].iloc[-1]}")
    print(f"  Kolom output     : {list(df_final.columns)}")
    print(f"  Contoh 5 baris pertama:")
    print(df_final[["id", "text", "label", "has_slang", "has_abbrev"]].head().to_string())
    print(f"{SEP_MAIN}")

    return df_final


### **Run Code**

In [ ]:
df_mixed = run_pipeline_mixed(
    input_path  = '/content/drive/MyDrive/Colab Datasets/combined_augmented_codemixed_dataset.csv',
    output_path = '/content/drive/MyDrive/Colab Datasets/dataset_mixed_final.csv',
    text_col    = 'text',
    label_col   = 'label',
    slang_col   = 'has_slang',
    abbrev_col  = 'has_abbrev',
    source      = 'generated',
    seed        = 42
)



  PIPELINE: CODE-MIXED

[1] LOAD RAW DATA
    File            : /content/drive/MyDrive/Colab Datasets/combined_augmented_codemixed_dataset.csv
    Baris dimuat    :  17,200
    Kolom           : ['id', 'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source']
    Hapus missing (text/label): 0
    Hapus non-binary: 0
    Hapus kosong    : 0
    Distribusi label: {0: 8394, 1: 8806}
    has_slang (LLM) : 0 (0.0%)
    has_abbrev (LLM): 0 (0.0%)
    Total valid     : 17,200 baris

[2] MINIMAL CLEANING (encoding fix + normalisasi whitespace)


    minimal_clean  : 100%|██████████| 17200/17200 [00:01<00:00, 16816.92it/s]


    Dilakukan pada  : 17,200 baris
    Contoh output   : 'gue literally belum makan dr tadi, starving bgt dan nobody even noticed'

[3] PENGECEKAN & KOREKSI FITUR (has_slang | has_abbrev)
    Koreksi: LLM=0 -> 1 jika word list mendeteksi (false negative).
    LLM=1 tetap 1 walaupun word list tidak menemukan (trust LLM).


    check_abbrev   : 100%|██████████| 17200/17200 [00:00<00:00, 21466.33it/s]



    [has_slang] — sebelum koreksi
      LLM=1, WL=1 (confirmed)    : 0
      LLM=1, WL=0 (LLM-only)     : 0  <- di luar word list, tetap 1
      LLM=0, WL=1 (false neg LLM): 8,347  <- akan dikoreksi ke 1
      LLM=0, WL=0 (neither)       : 8,853

    [has_abbrev] — sebelum koreksi
      LLM=1, WL=1 (confirmed)    : 0
      LLM=1, WL=0 (LLM-only)     : 0  <- di luar word list, tetap 1
      LLM=0, WL=1 (false neg LLM): 9,987  <- akan dikoreksi ke 1
      LLM=0, WL=0 (neither)       : 7,213

    Koreksi has_slang (0->1)  : 8,347 baris
    Koreksi has_abbrev (0->1) : 9,987 baris

    Distribusi setelah koreksi:
    has_slang        : 8,347 (48.5%)
    has_abbrev       : 9,987 (58.1%)
    (slang, abbrev)  : (1,1)=5486  (1,0)=2861  (0,1)=4501  (0,0)=4352

[4] DISTRIBUTION ANALYSIS (dicatat untuk Bab 3)
    -------------------------------------------------------
    NON-HATE (n= 8,394): slang=6586(78.5%)  abbrev=4706(56.1%)  (1,1)=4125  (0,0)=1227
    HATE    (n= 8,806): slang=1761(20.0%)  

         clean_text     : 100%|██████████| 17190/17190 [00:01<00:00, 9747.44it/s] 


         Selesai               : 17,190 baris
    [5c] Filter kosong + dedup   : 17,190 => 17,173 (hapus 17)
    [5d] Filter panjang (3-512)  : 17,173 => 17,173 (hapus 0)

[6] POST-CLEANING FILTER + VERIFIKASI DISTRIBUSI
    Total baris final: 17,173
    Distribusi label : {0: 8387, 1: 8786}

    Verifikasi drift distribusi (threshold delta > 5%)
    -------------------------------------------------------
    Label=0: pre=0.4880 => post=0.4884  delta=0.0004 ok
    Label=1: pre=0.5120 => post=0.5116  delta=0.0004 ok
    has_slang : pre=0.4853 => post=0.4858  delta=0.0005 ok
    has_abbrev: pre=0.5806 => post=0.5811  delta=0.0004 ok
    -------------------------------------------------------
    ok  Tidak ada drift signifikan. Distribusi terjaga dengan baik.

    TABEL TRANSISI DATA (Bab 3 -- Tabel transisi_cleaning)
    -------------------------------------------------------
    [1] Data mentah (raw CSV)                         :  17,200
    [2] Setelah validasi (NaN / kosong / non-bina